# Lifetime Model - PyTorch Distributed con TorchDistributor

**Author:** DLHub

**Date:** 12/05/2026


Version con `ArrowParquetBatchDataset` lazy, z-score en Spark y debug SSL en workers.

**Actualizacion:** entrenamiento DDP lee Parquet desde cache local por rank/worker; boto3 descarga desde S3 para evitar errores SSL de PyArrow S3FileSystem.


# 0. Instalacion de paquetes en ADA

Version adaptada para entrenamiento distribuido con `run_with_torch_distributor`, manteniendo lectura streaming con PyArrow para no materializar la data completa en memoria.



In [ ]:
%load_ext sagemaker_studio_analytics_extension.magics
%sm_analytics emr connect  \
    --cluster-id j-3PKXCEAPBCUQJ \
    --auth-type Basic_Access \
    --language python \
    --emr-execution-role-arn arn:aws:iam::430118852718:role/BBVARole-ada-us-east-1-sbx-live-s-size-dev

In [ ]:
%pip install scikit-learn pandas awswrangler s3fs pyarrow -q


# 1. Libraries

In [ ]:
import os

# Debe ejecutarse antes de importar PyArrow en EMR/Livy para evitar errores SSL curlCode 60.
CA_BUNDLE = os.getenv("CA_BUNDLE", "/etc/pki/ca-trust/extracted/pem/tls-ca-bundle.pem")
os.environ["SSL_CERT_FILE"] = CA_BUNDLE
os.environ["AWS_CA_BUNDLE"] = CA_BUNDLE
os.environ["CURL_CA_BUNDLE"] = CA_BUNDLE
os.environ["REQUESTS_CA_BUNDLE"] = CA_BUNDLE
os.environ["ARROW_SSL_CERT_FILE"] = CA_BUNDLE

import hashlib
from dataclasses import dataclass
from contextlib import nullcontext
from typing import Optional, Iterable

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.fs as pafs
import torch
import torch.distributed as dist
import torch.nn as nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

from mercury.contrib.ml.nn.torch import run_with_torch_distributor


In [ ]:
from dataproc_sdk.dataproc_sdk_datiopysparksession.datiopysparksession import DatioPysparkSession

In [ ]:
dataproc = DatioPysparkSession().get_or_create()
spark = dataproc.getSparkSession()

# 2. Config & Constants

# 2.1 Explorando Datos para train y test

In [ ]:
TRAIN_PATH = "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente"

In [ ]:
TEST_PATH = "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente"

In [ ]:
%%bash
aws s3 ls s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente --recursive | awk -F"/" '{print $4}' | sort | uniq -c

El comando previo arroja las siguientes particiones
```
1 _SUCCESS
    117 cutoff_date=2024-11-01
    266 cutoff_date=2024-12-01
    368 cutoff_date=2025-01-01
    414 cutoff_date=2025-02-01
    415 cutoff_date=2025-03-01
    386 cutoff_date=2025-04-01
    268 cutoff_date=2025-05-01
    250 cutoff_date=2025-06-01
    350 cutoff_date=2025-07-01
    255 cutoff_date=2025-08-01
```

In [ ]:
%%bash
aws s3 ls s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente --recursive | awk -F"/" '{print $4}' | sort | uniq -c

Estas son las particiones en la segunda ruta:
```
      1 _SUCCESS
    190 cutoff_date=2025-09-01
    112 cutoff_date=2025-10-01
    313 cutoff_date=2025-11-01
    306 cutoff_date=2025-12-01
```

## 2.2 Paths que se usarán train y test

In [ ]:
# Usaremos particiones de 6 meses para entrenar
PATHS = [
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente/cutoff_date=2025-03-01/",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente/cutoff_date=2025-04-01/",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente/cutoff_date=2025-05-01/",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente/cutoff_date=2025-06-01/",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente/cutoff_date=2025-07-01/",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente/cutoff_date=2025-08-01/",
]


# Aqui consideramos todas las particiones
VALID_PATHS = [
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente/cutoff_date=2025-09-01/",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente/cutoff_date=2025-10-01/",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente/cutoff_date=2025-11-01/",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente/cutoff_date=2025-12-01/",
]

## 2.3 Configuraciones para el modelo

In [ ]:
TARGET_COL = "independent"
ID_COL = "customer_id"

NUMERIC_COLS = [
    "transct_size",
    "product_size",
    "independent_unmes",
    "avg_transactions_amount_imputed",
    "bhsc_seg_score",
    "total_transaction_number_imputed",
    "cpac_average_balance_amount_med",
    "account_esntl_expenses_amount_imputed",
    "non_credit_incm_total_amount_imputed",
    "avg_essential_ticket",
    "account_nesl_expenses_amount_imputed",
    "seniority",
    "token_score",
    "essential_expense_ratio",
    "income_score",
    "age_clean",
    "month_cpac_srty_max_number_imputed",
    "non_essential_ratio",
    "bhsc_mdl_cust_tot_scor_number_med",
    "segment_score",
    "incm_uniq_model_result_number_med",
    "foodie_unmes",
    "avg_captation_balance_amount_med",
    "title_score",
    "young_and_single",
    "os_score",
    "seguro_salud",
    "student_unmes",
    "seguro_salud_viaje",
    "is_married_like",
]

AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")

HASH_BUCKET_SIZE = int(os.getenv("HASH_BUCKET_SIZE", str(2**18)))
EMBEDDING_DIM = int(os.getenv("EMBEDDING_DIM", "16"))

# Arrow lee en batches grandes desde S3. El DataLoader NO vuelve a micro-batchear.
# El dataset produce directamente tensores de tamaño TRAIN_BATCH_SIZE.
BATCH_SIZE_ROWS = int(os.getenv("BATCH_SIZE_ROWS", "200000"))
TRAIN_BATCH_SIZE = int(os.getenv("TRAIN_BATCH_SIZE", "8192"))
NUM_WORKERS = int(os.getenv("NUM_WORKERS", "0"))
PIN_MEMORY = False

# Cache local por rank/worker para evitar PyArrow S3FileSystem durante entrenamiento.
# Boto3 descarga cada Parquet asignado al worker y PyArrow lo lee desde disco local.
PARQUET_LOCAL_CACHE_DIR = os.getenv("PARQUET_LOCAL_CACHE_DIR", "/tmp/lifetime_parquet_cache")

EPOCHS = int(os.getenv("EPOCHS", "3"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE", "1e-3"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "1e-4"))

FILL_NULL_VALUE = 0.0
SHUFFLE_WITHIN_BATCH = os.getenv("SHUFFLE_WITHIN_BATCH", "true").lower() in {"1", "true", "yes", "y"}

DEVICE = "cpu"
SEED = int(os.getenv("SEED", "12345"))

# Configuracion fija para EMR CPU distribuido.
# Los nodos no tienen GPU, asi que usamos gloo y local_mode=False.
USE_GPU = False
NUM_PROCESSES = int(os.getenv("NUM_PROCESSES", "2"))
LOCAL_MODE = False
DISTRIBUTED_BACKEND = "gloo"

MODEL_ARTIFACT_PATH = os.getenv("MODEL_ARTIFACT_PATH", "/tmp/lifetime_tabular_ddp.pt")

print(
    "Config DDP:",
    {
        "USE_GPU": USE_GPU,
        "NUM_PROCESSES": NUM_PROCESSES,
        "LOCAL_MODE": LOCAL_MODE,
        "BACKEND": DISTRIBUTED_BACKEND,
        "BATCH_SIZE_ROWS": BATCH_SIZE_ROWS,
        "TRAIN_BATCH_SIZE": TRAIN_BATCH_SIZE,
        "NUM_WORKERS": NUM_WORKERS,
    },
)


# Spark prepara datasets una sola vez por split. PyTorch/DDP solo entrena sobre estos Parquet ya limpios.
RUN_SPARK_PREPARE = os.getenv("RUN_SPARK_PREPARE", "true").lower() in {"1", "true", "yes", "y"}
SPARK_WRITE_MODE = os.getenv("SPARK_WRITE_MODE", "overwrite")
SPARK_REPARTITIONS_TRAIN = int(os.getenv("SPARK_REPARTITIONS_TRAIN", "512"))
SPARK_REPARTITIONS_VALID = int(os.getenv("SPARK_REPARTITIONS_VALID", "256"))
ID_BUCKET_COL = os.getenv("ID_BUCKET_COL", "customer_id_bucket")

PREPARED_BASE_PATH = os.getenv(
    "PREPARED_BASE_PATH",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/lifetime_prepared_ddp",
)
PREPARED_TRAIN_PATH = os.getenv("PREPARED_TRAIN_PATH", f"{PREPARED_BASE_PATH}/train_prepared/")
PREPARED_VALID_PATH = os.getenv("PREPARED_VALID_PATH", f"{PREPARED_BASE_PATH}/valid_prepared/")

# Logs por rank para depurar errores ocultos por Spark Barrier/TorchDistributor.
DEBUG_LOG_PATH = os.getenv(
    "DEBUG_LOG_PATH",
    f"{PREPARED_BASE_PATH.rstrip('/')}/debug_logs/",
)


# El hash Spark SQL de abajo usa los bits bajos del MD5. Esto coincide con int(md5, 16) % HASH_BUCKET_SIZE
# cuando HASH_BUCKET_SIZE es potencia de 2, como el default 2**18.
if HASH_BUCKET_SIZE <= 0 or (HASH_BUCKET_SIZE & (HASH_BUCKET_SIZE - 1)) != 0:
    raise ValueError("HASH_BUCKET_SIZE debe ser potencia de 2 para que el bucket MD5 de Spark sea equivalente al MD5 Python.")


## Preparación con Spark y entrenamiento DDP sobre Parquet preparado

Esta etapa usa Spark para leer la data cruda, aplicar selección de columnas, casteos, `fillna`, calcular el bucket MD5 de `customer_id`, **calcular el z-score con estadísticas de train y escribir las columnas numéricas ya estandarizadas**, reparticionar y escribir Parquet preparado. El entrenamiento DDP posterior lee esos archivos preparados con PyArrow, evitando ETL repetido en cada época.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T


def get_or_create_spark():
    """Usa la SparkSession de Livy/EMR si ya existe; si no, crea una local."""
    try:
        return spark
    except NameError:
        from pyspark.sql import SparkSession
        return SparkSession.builder.appName("lifetime-ddp-preparation").getOrCreate()


def md5_bucket_expr(col_name: str, num_buckets: int):
    """
    Bucket MD5 compatible con stable_hash_bucket(value, 2**k).

    Spark conv(hex, 16, 10) puede desbordar con 32 chars hex; por eso usamos los 15 chars finales
    del MD5. Para modulo potencia de 2, bastan los bits bajos y el resultado coincide con:
        int(hashlib.md5(str(value).encode('utf-8')).hexdigest(), 16) % num_buckets
    """
    low_hex = F.expr(f"right(md5(coalesce(cast({col_name} as string), '__MISSING__')), 15)")
    return F.pmod(F.conv(low_hex, 16, 10).cast("long"), F.lit(int(num_buckets))).cast("long")


def _read_numeric_frame_for_stats(spark_session, input_paths: list[str]):
    """Lee columnas numericas raw, castea y aplica fillna igual que el preprocesamiento final."""
    df = spark_session.read.parquet(*input_paths).select(*NUMERIC_COLS)
    for c in NUMERIC_COLS:
        df = df.withColumn(c, F.col(c).cast("double"))
    return df.fillna(float(FILL_NULL_VALUE), subset=NUMERIC_COLS)


def compute_spark_standardization_stats(input_paths: list[str]) -> tuple[np.ndarray, np.ndarray]:
    """
    Calcula mean/std en Spark usando SOLO train raw.

    Estas estadisticas se usan para estandarizar train y valid durante el preprocesamiento Spark.
    El entrenamiento DDP ya no vuelve a hacer z-score por batch.
    """
    spark_session = get_or_create_spark()
    df = _read_numeric_frame_for_stats(spark_session, input_paths)

    agg_exprs = []
    for c in NUMERIC_COLS:
        agg_exprs.append(F.avg(F.col(c)).alias(f"{c}__mean"))
        agg_exprs.append(F.stddev_pop(F.col(c)).alias(f"{c}__std"))

    row = df.agg(*agg_exprs).collect()[0]

    means = []
    stds = []
    for c in NUMERIC_COLS:
        mean_value = row[f"{c}__mean"]
        std_value = row[f"{c}__std"]

        if mean_value is None:
            mean_value = 0.0
        if std_value is None or float(std_value) < 1e-8:
            std_value = 1.0

        means.append(float(mean_value))
        stds.append(float(std_value))

    scaler_mean = np.asarray(means, dtype=np.float32)
    scaler_std = np.asarray(stds, dtype=np.float32)

    print("Scaler Spark calculado sobre train raw.")
    print(f"Features: {len(NUMERIC_COLS)}")
    print(f"Primeras medias: {scaler_mean[:5]}")
    print(f"Primeras stds: {scaler_std[:5]}")

    return scaler_mean, scaler_std


def prepare_lifetime_split_with_spark(
    input_paths: list[str],
    output_path: str,
    repartitions: int,
    split_name: str,
    scaler_mean: np.ndarray,
    scaler_std: np.ndarray,
):
    """
    Lee Parquet raw con Spark, prepara columnas y escribe Parquet optimizado para DDP.

    Importante:
    - El z-score de NUMERIC_COLS se hace aqui, durante el preprocesamiento.
    - El dataset PyArrow de entrenamiento solo lee tensores ya estandarizados.
    - Las estadisticas de estandarizacion deben venir de train y se aplican tambien a valid.
    """
    spark_session = get_or_create_spark()

    required_raw_cols = list(dict.fromkeys(NUMERIC_COLS + [TARGET_COL, ID_COL]))

    print(f"[{split_name}] Leyendo raw paths:")
    for p in input_paths:
        print(f"  - {p}")

    df = spark_session.read.parquet(*input_paths).select(*required_raw_cols)

    # Casteos explicitos para que PyArrow/PyTorch reciban tipos estables.
    for c in NUMERIC_COLS:
        df = df.withColumn(c, F.col(c).cast("double"))

    df = df.withColumn(TARGET_COL, F.col(TARGET_COL).cast("int"))

    # Filtros minimos de calidad. Agrega aqui filtros de negocio si quieres restringir poblacion.
    df = df.filter(F.col(TARGET_COL).isNotNull())

    # Fillna solo en variables numericas; target nulo ya fue filtrado.
    df = df.fillna(float(FILL_NULL_VALUE), subset=NUMERIC_COLS)

    # Z-score en Spark. Mantiene los mismos nombres de columnas numericas.
    for idx, c in enumerate(NUMERIC_COLS):
        mean_value = float(scaler_mean[idx])
        std_value = float(scaler_std[idx])
        if std_value < 1e-8:
            std_value = 1.0
        df = df.withColumn(
            c,
            ((F.col(c) - F.lit(mean_value)) / F.lit(std_value)).cast("float"),
        )

    # Precalculo de MD5 bucket para no pagar hashlib en cada epoca/rank/worker.
    df = df.withColumn(ID_BUCKET_COL, md5_bucket_expr(ID_COL, HASH_BUCKET_SIZE))

    prepared_cols = NUMERIC_COLS + [TARGET_COL, ID_BUCKET_COL]
    df = df.select(*prepared_cols)

    if repartitions and repartitions > 0:
        df = df.repartition(int(repartitions))

    print(f"[{split_name}] Escribiendo Parquet preparado y estandarizado en: {output_path}")
    (
        df.write
        .mode(SPARK_WRITE_MODE)
        .option("compression", "snappy")
        .parquet(output_path)
    )

    print(f"[{split_name}] Preparacion Spark terminada.")
    return output_path


def prepare_lifetime_datasets_with_spark():
    """
    Prepara train/valid con Spark y devuelve las estadisticas raw de train.

    La data escrita en PREPARED_TRAIN_PATH/PREPARED_VALID_PATH ya queda z-scoreada.
    """
    scaler_mean, scaler_std = compute_spark_standardization_stats(PATHS)

    train_out = prepare_lifetime_split_with_spark(
        input_paths=PATHS,
        output_path=PREPARED_TRAIN_PATH,
        repartitions=SPARK_REPARTITIONS_TRAIN,
        split_name="train",
        scaler_mean=scaler_mean,
        scaler_std=scaler_std,
    )
    valid_out = prepare_lifetime_split_with_spark(
        input_paths=VALID_PATHS,
        output_path=PREPARED_VALID_PATH,
        repartitions=SPARK_REPARTITIONS_VALID,
        split_name="valid",
        scaler_mean=scaler_mean,
        scaler_std=scaler_std,
    )
    return train_out, valid_out, scaler_mean, scaler_std

# 3. Auxiliar Functions
## 3.1 Utilities

In [ ]:
def set_seed(seed: int) -> None:
    """
    Configura la semilla de aleatoriedad para NumPy y PyTorch.

    Establece un estado determinista en las librerías de generación de números 
    aleatorios, incluyendo el soporte para GPU (CUDA) si está disponible.

    Args:
        seed (int): El valor numérico de la semilla a utilizar.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def stable_hash_bucket(value: str, num_buckets: int) -> int:
    """
        Asigna una cadena de texto a un contenedor (bucket) de forma determinista.

        Utiliza el algoritmo MD5 para garantizar que un mismo valor siempre caiga 
        en el mismo bucket, incluso entre diferentes sesiones de Python.

        Args:
            value (str): La cadena de texto a procesar. Si es None, se usa un valor por defecto.
            num_buckets (int): El número total de contenedores disponibles.

        Returns:
            int: El índice del bucket asignado, calculado como $hash \pmod n$.
    """
    if value is None:
        value = "__MISSING__"
    digest = hashlib.md5(str(value).encode("utf-8")).hexdigest()
    return int(digest, 16) % num_buckets



def configure_worker_runtime() -> None:
    """Configura variables necesarias dentro de cada worker de TorchDistributor.

    En EMR/Livy, las variables seteadas en el driver no siempre llegan al proceso
    Python que ejecuta cada rank. Esta funcion debe llamarse dentro de train_fn
    y tambien antes de crear S3FileSystem en el IterableDataset.
    """
    ca_bundle = os.environ.get("CA_BUNDLE", "/etc/pki/ca-trust/extracted/pem/tls-ca-bundle.pem")
    os.environ["SSL_CERT_FILE"] = ca_bundle
    os.environ["AWS_CA_BUNDLE"] = ca_bundle
    os.environ["CURL_CA_BUNDLE"] = ca_bundle
    os.environ["REQUESTS_CA_BUNDLE"] = ca_bundle
    os.environ["ARROW_SSL_CERT_FILE"] = ca_bundle


def get_s3_filesystem(region: str = AWS_REGION) -> pafs.S3FileSystem:
    """Crea S3FileSystem compatible con la version de PyArrow en EMR/Livy."""
    configure_worker_runtime()
    return pafs.S3FileSystem(region=region)


def write_text_to_s3(text: str, s3_uri: str, region: str = AWS_REGION) -> str:
    """Escribe texto a S3 usando boto3. Devuelve la URI escrita."""
    import boto3

    bucket, key = parse_s3_uri(s3_uri)
    client = boto3.client("s3", region_name=region)
    client.put_object(Bucket=bucket, Key=key, Body=text.encode("utf-8"))
    return s3_uri


def write_rank_debug_log(text: str, rank: str | int = "unknown", region: str = AWS_REGION) -> str | None:
    """Escribe un log por rank en S3. Si S3 falla, escribe fallback local en /tmp."""
    import datetime
    import os

    ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S_%f")
    safe_rank = str(rank).replace("/", "_")
    base = DEBUG_LOG_PATH.rstrip("/")
    log_uri = f"{base}/rank_{safe_rank}_{ts}.log"

    try:
        written = write_text_to_s3(text=text, s3_uri=log_uri, region=region)
        print(f"Debug log escrito en {written}", flush=True)
        return written
    except Exception as log_error:
        fallback = f"/tmp/lifetime_ddp_rank_{safe_rank}_{ts}.log"
        try:
            with open(fallback, "w", encoding="utf-8") as f:
                f.write(text)
            print(f"No se pudo escribir log a S3: {repr(log_error)}", flush=True)
            print(f"Debug log local escrito en {fallback}", flush=True)
        except Exception as local_error:
            print(f"No se pudo escribir log local: {repr(local_error)}", flush=True)
        return None


def format_worker_diagnostics(prefix: str = "") -> str:
    """Devuelve diagnostico basico del worker/rank actual."""
    import os
    import platform
    import socket

    keys = [
        "RANK", "LOCAL_RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT",
        "PYSPARK_TASK_ATTEMPT_ID", "CUDA_VISIBLE_DEVICES", "SSL_CERT_FILE",
        "AWS_CA_BUNDLE", "ARROW_SSL_CERT_FILE", "CA_BUNDLE",
    ]
    lines = []
    if prefix:
        lines.append(prefix)
    lines.extend([
        f"host={socket.gethostname()}",
        f"pid={os.getpid()}",
        f"python={platform.python_version()}",
        f"torch={getattr(torch, '__version__', 'unknown')}",
        f"pyarrow={getattr(pa, '__version__', 'unknown')}",
    ])
    for k in keys:
        lines.append(f"{k}={os.environ.get(k)}")
    return "\n".join(lines)


def extract_worker_paths(paths: list[str]) -> list[str]:
    """
        Divide una lista de rutas entre los diferentes trabajadores (workers) activos.

        Utiliza una técnica de 'striding' (salto) basada en el ID del trabajador 
        para asegurar que cada proceso reciba un subconjunto único de archivos, 
        evitando duplicación de datos en procesos paralelos.

        Args:
            paths (list[str]): Lista completa de rutas de archivos a procesar.

        Returns:
            list[str]: Subconjunto de rutas asignadas específicamente al trabajador actual. 
                Si no hay información de workers, devuelve la lista original.
    """
    worker_info = get_worker_info()
    if worker_info is None:
        return paths
    return paths[worker_info.id::worker_info.num_workers]


def parse_s3_uri(uri: str) -> tuple[str, str]:
    """
    Convierte:
        s3://bucket/key...
    en:
        ("bucket", "key...")
    """
    if not uri.startswith("s3://"):
        raise ValueError(f"Se esperaba URI s3://..., recibido: {uri}")
    no_scheme = uri[5:]
    parts = no_scheme.split("/", 1)
    bucket = parts[0]
    key = parts[1] if len(parts) > 1 else ""
    return bucket, key


def normalize_s3_path(path: str) -> str:
    """
    Convierte:
      - s3://bucket/key
      - bucket/key
    a formato PyArrow S3FileSystem:
      - bucket/key
    """
    if path.startswith("s3://"):
        bucket, key = parse_s3_uri(path)
        return f"{bucket}/{key}".rstrip("/")
    return path.rstrip("/")


def expand_s3_paths(paths: list[str], region: str = AWS_REGION) -> list[str]:
    """
    Expande prefijos S3 a archivos parquet reales.
    Admite:
      - s3://bucket/prefix/
      - s3://bucket/file.parquet
      - bucket/prefix/
      - bucket/file.parquet

    Devuelve una lista de rutas tipo:
      bucket/key/file.parquet
    """
    fs = get_s3_filesystem(region=region)
    out = []

    for raw_path in paths:
        path = normalize_s3_path(raw_path)

        if "*" in path:
            raise ValueError(
                f"No uses globs en PATHS con PyArrow S3FileSystem: {raw_path}\n"
                f"Usa el prefijo del directorio, por ejemplo .../cutoff_date=2025-08-01/"
            )

        if path.endswith(".parquet"):
            info = fs.get_file_info([path])[0]
            if info.type == pafs.FileType.File:
                out.append(path)
            else:
                raise FileNotFoundError(f"No existe el archivo parquet: {raw_path}")
            continue

        selector = pafs.FileSelector(path, recursive=True)
        infos = fs.get_file_info(selector)

        parquet_files = [
            info.path
            for info in infos
            if info.type == pafs.FileType.File and info.path.endswith(".parquet")
        ]

        if not parquet_files:
            raise FileNotFoundError(f"No se encontraron .parquet bajo el prefijo: {raw_path}")

        out.extend(sorted(parquet_files))

    if not out:
        raise ValueError("No se resolvió ningún archivo parquet.")

    return out


def get_dist_info() -> tuple[int, int, int]:
    """
    Lee la configuracion de proceso que inyecta TorchDistributor.
    Devuelve: rank global, local_rank y world_size.
    """
    rank = int(os.environ.get("RANK", os.environ.get("OMPI_COMM_WORLD_RANK", "0")))
    local_rank = int(os.environ.get("LOCAL_RANK", os.environ.get("OMPI_COMM_WORLD_LOCAL_RANK", "0")))
    world_size = int(os.environ.get("WORLD_SIZE", os.environ.get("OMPI_COMM_WORLD_SIZE", "1")))
    return rank, local_rank, world_size


def setup_distributed(backend: str = DISTRIBUTED_BACKEND) -> tuple[int, int, int]:
    rank, local_rank, world_size = get_dist_info()
    if world_size > 1 and not dist.is_initialized():
        dist.init_process_group(backend=backend)
    return rank, local_rank, world_size


def cleanup_distributed() -> None:
    if dist.is_available() and dist.is_initialized():
        try:
            dist.barrier()
        except Exception:
            pass
        try:
            dist.destroy_process_group()
        except Exception:
            pass


def select_device(local_rank: int, use_gpu: bool = USE_GPU) -> torch.device:
    if use_gpu and torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        return torch.device(f"cuda:{local_rank}")
    return torch.device("cpu")


def reduce_sum(value: float, device: torch.device) -> float:
    tensor = torch.tensor(value, dtype=torch.float64, device=device)
    if dist.is_available() and dist.is_initialized():
        dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    return float(tensor.item())


def extract_parallel_paths(paths: list[str]) -> list[str]:
    """
    Divide una lista de ARCHIVOS parquet ya resueltos primero por rank DDP
    y despues por worker del DataLoader.

    Importante: no pasar aqui un unico prefijo S3, porque eso no reparte
    trabajo. El driver debe resolver prefijos a archivos con expand_s3_paths()
    antes de lanzar run_with_torch_distributor.
    """
    rank, _, world_size = get_dist_info()
    paths_for_rank = sorted(paths)[rank::world_size]

    worker_info = get_worker_info()
    if worker_info is None:
        return paths_for_rank
    return paths_for_rank[worker_info.id::worker_info.num_workers]


def batched_indices(n_rows: int, batch_size: int, shuffle: bool) -> Iterable[np.ndarray]:
    indices = np.arange(n_rows)
    if shuffle:
        np.random.shuffle(indices)
    for start in range(0, n_rows, batch_size):
        yield indices[start:start + batch_size]


# 3.2 Streaming & others

In [ ]:
class StreamingStandardizer:
    """
    Calculador incremental de media y desviacion estandar.
    """
    def __init__(self, n_features: int):
        self.n = 0
        self.sum = np.zeros(n_features, dtype=np.float64)
        self.sum_sq = np.zeros(n_features, dtype=np.float64)

    def update(self, x: np.ndarray) -> None:
        if x.size == 0:
            return
        self.n += x.shape[0]
        self.sum += x.sum(axis=0)
        self.sum_sq += np.square(x).sum(axis=0)

    def finalize(self) -> tuple[np.ndarray, np.ndarray]:
        if self.n == 0:
            raise ValueError("No se encontraron filas para calcular mean/std.")
        mean = self.sum / self.n
        var = self.sum_sq / self.n - np.square(mean)
        var = np.maximum(var, 1e-12)
        std = np.sqrt(var)
        return mean.astype(np.float32), std.astype(np.float32)


def iter_arrow_batches(
    paths: list[str],
    columns: list[str],
    batch_size_rows: int,
    region: str = AWS_REGION,
) -> Iterable[pa.RecordBatch]:
    """
    Lee Parquet desde S3 con PyArrow Dataset y devuelve RecordBatch.

    PyArrow mantiene lectura por columnas, batches y pushdown cuando aplica.
    Esto evita materializar el dataset completo y elimina la dependencia de Polars
    en el camino caliente del entrenamiento.
    """
    filesystem = get_s3_filesystem(region=region)
    resolved_files = expand_s3_paths(paths, region=region)

    dataset = ds.dataset(
        resolved_files,
        format="parquet",
        filesystem=filesystem,
    )

    scanner = dataset.scanner(
        columns=columns,
        batch_size=batch_size_rows,
        use_threads=True,
    )

    yield from scanner.to_batches()


def arrow_numeric_matrix(record_batch: pa.RecordBatch, numeric_cols: list[str], fill_null_value: float) -> np.ndarray:
    """
    Convierte columnas numericas de un RecordBatch a matriz float32.
    Se hace columna a columna para mantener memoria acotada.
    """
    cols = []
    for col_name in numeric_cols:
        arr = record_batch.column(col_name)
        arr = pc.cast(arr, pa.float32(), safe=False)
        arr = pc.fill_null(arr, fill_null_value)
        cols.append(arr.to_numpy(zero_copy_only=False))

    if not cols:
        raise ValueError("numeric_cols esta vacio.")

    return np.column_stack(cols).astype(np.float32, copy=False)


def arrow_target_vector(
    record_batch: pa.RecordBatch,
    target_col: str,
    fill_null_value: float = 0.0,
    dtype: pa.DataType = pa.float32(),
) -> np.ndarray:
    arr = record_batch.column(target_col)
    arr = pc.cast(arr, dtype, safe=False)
    arr = pc.fill_null(arr, fill_null_value)
    return arr.to_numpy(zero_copy_only=False)


def md5_to_bucket(values, modulo: int) -> np.ndarray:
    """MD5 exacto en Python. Se deja como fallback si lees raw Parquet sin ID_BUCKET_COL."""
    return np.fromiter(
        (
            int(hashlib.md5(("__MISSING__" if v is None else str(v)).encode("utf-8")).hexdigest(), 16) % modulo
            for v in values
        ),
        dtype=np.int64,
        count=len(values),
    )


def arrow_hash_id_vector(record_batch: pa.RecordBatch, id_col: str, hash_bucket_size: int) -> np.ndarray:
    """
    Fallback con MD5 exacto cuando el Parquet no trae ID_BUCKET_COL precalculado.
    No usa pandas.util.hash_pandas_object.
    """
    arr = record_batch.column(id_col)
    arr = pc.cast(arr, pa.string(), safe=False)
    values = arr.to_pylist()
    return md5_to_bucket(values, hash_bucket_size)


def arrow_id_bucket_vector(record_batch: pa.RecordBatch, id_bucket_col: str) -> np.ndarray:
    """Lee el bucket MD5 precalculado por Spark."""
    arr = record_batch.column(id_bucket_col)
    arr = pc.cast(arr, pa.int64(), safe=False)
    arr = pc.fill_null(arr, 0)
    return arr.to_numpy(zero_copy_only=False).astype(np.int64, copy=False)

def compute_streaming_stats(
    paths: list[str],
    numeric_cols: list[str],
    batch_size_rows: int = BATCH_SIZE_ROWS,
    fill_null_value: float = FILL_NULL_VALUE,
    region: str = AWS_REGION,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Calcula mean/std en streaming usando solo PyArrow + NumPy.
    """
    stats = StreamingStandardizer(len(numeric_cols))

    for record_batch in iter_arrow_batches(paths=paths, columns=numeric_cols, batch_size_rows=batch_size_rows, region=region):
        x = arrow_numeric_matrix(record_batch, numeric_cols, fill_null_value)
        stats.update(x)

    return stats.finalize()



In [ ]:
def split_s3_location(path: str) -> tuple[str, str]:
    """
    Acepta tanto:
      - s3://bucket/key
      - bucket/key

    Devuelve:
      (bucket, key)
    """
    if path.startswith("s3://"):
        return parse_s3_uri(path)

    clean_path = path.lstrip("/")
    parts = clean_path.split("/", 1)
    if len(parts) != 2 or not parts[0] or not parts[1]:
        raise ValueError(f"Ruta S3 invalida para descarga boto3: {path}")
    return parts[0], parts[1]


def local_cache_path_for_s3_path(s3_path: str, cache_dir: str) -> str:
    """Construye un nombre local estable para un objeto S3."""
    bucket, key = split_s3_location(s3_path)
    digest = hashlib.md5(f"s3://{bucket}/{key}".encode("utf-8")).hexdigest()
    filename = os.path.basename(key) or "part.parquet"
    return os.path.join(cache_dir, f"{digest}_{filename}")


def download_s3_to_local_if_needed(
    s3_path: str,
    local_path: str,
    region: str = AWS_REGION,
) -> str:
    """
    Descarga un objeto S3 con boto3 si no existe en cache local.

    Esto evita pyarrow.fs.S3FileSystem dentro de los workers de TorchDistributor,
    que en este entorno falla con curlCode 60 durante HeadObject/ListObjectsV2.
    """
    import boto3

    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        return local_path

    bucket, key = split_s3_location(s3_path)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    tmp_path = f"{local_path}.tmp.{os.getpid()}"
    if os.path.exists(tmp_path):
        os.remove(tmp_path)

    client = boto3.client("s3", region_name=region)
    client.download_file(bucket, key, tmp_path)
    os.replace(tmp_path, local_path)

    return local_path


class ArrowParquetBatchDataset(IterableDataset):
    """
    Dataset lazy para Parquet preparado por Spark.

    - Spark escribe Parquet preparado y z-scoreado en S3.
    - Cada rank/worker recibe un shard de archivos.
    - Boto3 descarga esos archivos a cache local.
    - PyArrow lee Parquet local, no S3. Esto evita el error SSL de PyArrow S3FileSystem.
    - Produce mini-batches tensorizados como diccionarios: x_num, x_id, y.
    - Usa customer_id_bucket ya precalculado por Spark, sin recalcular MD5 en cada epoca.

    Nota: este dataset ya produce batches; por eso el DataLoader debe usar batch_size=None.
    """

    def __init__(
        self,
        files: list[str],
        numeric_cols: list[str],
        id_bucket_col: str,
        target_col: str,
        batch_size_rows: int = BATCH_SIZE_ROWS,
        train_batch_size: int = TRAIN_BATCH_SIZE,
        fill_null_value: float = FILL_NULL_VALUE,
        shuffle_within_batch: bool = False,
        region: str = AWS_REGION,
        rank: int = 0,
        world_size: int = 1,
        cache_dir: str = PARQUET_LOCAL_CACHE_DIR,
    ):
        super().__init__()
        self.files = sorted(files)
        self.numeric_cols = list(numeric_cols)
        self.id_bucket_col = id_bucket_col
        self.target_col = target_col
        self.batch_size_rows = int(batch_size_rows)
        self.train_batch_size = int(train_batch_size)
        self.fill_null_value = fill_null_value
        self.shuffle_within_batch = shuffle_within_batch
        self.region = region
        self.rank = int(rank)
        self.world_size = int(world_size)
        self.cache_dir = cache_dir
        self.required_cols = self.numeric_cols + [self.id_bucket_col, self.target_col]

    def _local_files_for_rank_and_worker(self) -> tuple[list[str], int, int]:
        worker_info = get_worker_info()
        worker_id = 0 if worker_info is None else worker_info.id
        num_workers = 1 if worker_info is None else worker_info.num_workers

        shard_count = max(self.world_size, 1) * num_workers
        shard_id = self.rank * num_workers + worker_id

        shard_files = self.files[shard_id::shard_count]

        print(
            f"[rank={self.rank}/{self.world_size} worker={worker_id}/{num_workers}] "
            f"asignados {len(shard_files)} de {len(self.files)} archivos parquet preparados",
            flush=True,
        )
        return shard_files, worker_id, num_workers

    def _download_shard_to_local_cache(self, shard_files: list[str], worker_id: int) -> list[str]:
        rank_cache_dir = os.path.join(
            self.cache_dir,
            f"rank_{self.rank}",
            f"worker_{worker_id}",
        )

        local_paths = []
        for i, s3_path in enumerate(shard_files, start=1):
            local_path = local_cache_path_for_s3_path(s3_path=s3_path, cache_dir=rank_cache_dir)
            downloaded = download_s3_to_local_if_needed(
                s3_path=s3_path,
                local_path=local_path,
                region=self.region,
            )
            local_paths.append(downloaded)

            if i == 1 or i % 50 == 0 or i == len(shard_files):
                print(
                    f"[rank={self.rank} worker={worker_id}] cache local {i}/{len(shard_files)}: {downloaded}",
                    flush=True,
                )

        return local_paths

    def __iter__(self):
        configure_worker_runtime()

        shard_files, worker_id, _ = self._local_files_for_rank_and_worker()
        if not shard_files:
            return

        local_files = self._download_shard_to_local_cache(shard_files, worker_id)

        dataset = ds.dataset(
            local_files,
            format="parquet",
        )
        scanner = dataset.scanner(
            columns=self.required_cols,
            batch_size=self.batch_size_rows,
            use_threads=True,
        )

        for record_batch in scanner.to_batches():
            # Ya viene z-scoreado desde Spark. Aqui solo convertimos a tensor float32.
            x_num = arrow_numeric_matrix(record_batch, self.numeric_cols, self.fill_null_value)
            x_id = arrow_id_bucket_vector(record_batch, self.id_bucket_col)
            y = arrow_target_vector(record_batch, self.target_col, fill_null_value=0.0).astype(np.float32, copy=False)

            for idx in batched_indices(record_batch.num_rows, self.train_batch_size, self.shuffle_within_batch):
                yield {
                    "x_num": torch.from_numpy(np.ascontiguousarray(x_num[idx])).float(),
                    "x_id": torch.from_numpy(np.ascontiguousarray(x_id[idx])).long(),
                    "y": torch.from_numpy(np.ascontiguousarray(y[idx])).float(),
                }


def make_train_loader(
    train_files: list[str],
    rank: int,
    world_size: int,
) -> DataLoader:
    train_dataset = ArrowParquetBatchDataset(
        files=train_files,
        numeric_cols=NUMERIC_COLS,
        id_bucket_col=ID_BUCKET_COL,
        target_col=TARGET_COL,
        batch_size_rows=BATCH_SIZE_ROWS,
        train_batch_size=TRAIN_BATCH_SIZE,
        fill_null_value=FILL_NULL_VALUE,
        shuffle_within_batch=SHUFFLE_WITHIN_BATCH,
        region=AWS_REGION,
        rank=rank,
        world_size=world_size,
        cache_dir=PARQUET_LOCAL_CACHE_DIR,
    )

    return DataLoader(
        train_dataset,
        batch_size=None,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        persistent_workers=(NUM_WORKERS > 0),
    )


def make_valid_loader(
    valid_files: list[str],
    rank: int,
    world_size: int,
) -> DataLoader:
    valid_dataset = ArrowParquetBatchDataset(
        files=valid_files,
        numeric_cols=NUMERIC_COLS,
        id_bucket_col=ID_BUCKET_COL,
        target_col=TARGET_COL,
        batch_size_rows=BATCH_SIZE_ROWS,
        train_batch_size=TRAIN_BATCH_SIZE,
        fill_null_value=FILL_NULL_VALUE,
        shuffle_within_batch=False,
        region=AWS_REGION,
        rank=rank,
        world_size=world_size,
        cache_dir=PARQUET_LOCAL_CACHE_DIR,
    )

    return DataLoader(
        valid_dataset,
        batch_size=None,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        persistent_workers=(NUM_WORKERS > 0),
    )


In [ ]:
def collate_tabular(batch):
    """
    Compatibilidad para versiones fila-a-fila.
    En el entrenamiento optimizado usamos DataLoader(..., batch_size=None),
    por lo que el dataset ya entrega dicts de tensores batchificados.
    """
    if isinstance(batch, dict):
        return batch
    if len(batch) == 1 and isinstance(batch[0], dict) and batch[0]["x_num"].ndim == 2:
        return batch[0]

    x_num = torch.stack([item["x_num"] for item in batch], dim=0)
    y = torch.stack([item["y"] for item in batch], dim=0)

    output = {"x_num": x_num, "y": y}
    if "x_id" in batch[0]:
        output["x_id"] = torch.stack([item["x_id"] for item in batch], dim=0)
    return output


In [ ]:
def compute_class_balance(
    paths: list[str],
    target_col: str,
    batch_size_rows: int = BATCH_SIZE_ROWS,
    region: str = AWS_REGION,
) -> tuple[int, int]:
    """
    Devuelve n_neg, n_pos calculados en streaming sobre train con PyArrow.
    """
    n_pos = 0
    n_neg = 0

    for record_batch in iter_arrow_batches(paths=paths, columns=[target_col], batch_size_rows=batch_size_rows, region=region):
        y = arrow_target_vector(record_batch, target_col, fill_null_value=0, dtype=pa.int64())
        n_pos += int((y == 1).sum())
        n_neg += int((y == 0).sum())

    return n_neg, n_pos



# 4. Model

Nota: En el modelo se usa un embedding que represente el `customer_id` usando una función hashing, dado que dicha variable categórica puede tener cardinalidad alta en producción (véase https://medium.com/nextgenllm/feature-hashing-efficient-categorical-data-encoding-for-large-scale-ml-systems-3d143a8ae239)

In [ ]:
class TabularNN(nn.Module):
    """
    Red Neuronal para datos tabulares con soporte para embeddings del customer_id.

    Esta arquitectura procesa simultáneamente variables numéricas y una variable
    categórica de alta cardinalidad (customer_id) mediante una capa de embedding.
    Ambas representaciones se concatenan y pasan a través de un Perceptrón
    Multicapa (MLP) con bloques de Linear-ReLU-BatchNorm-Dropout.

    Attributes:
        embedding (nn.Embedding): Capa que transforma los índices de hash en
            vectores densos (embeddings).
        mlp (nn.Sequential): Secuencia de capas densas y de regularización que
            realizan la regresión o clasificación.
    """

    def __init__(
        self,
        num_numeric_features: int,
        num_hash_buckets: int,
        emb_dim: int = EMBEDDING_DIM,
        hidden_dims: list[int] = [128, 64],
        dropout: float = 0.2,
    ):
        """
        Inicializa la arquitectura de la red.

        Args:
            num_numeric_features (int): Cantidad de columnas numéricas de entrada.
            num_hash_buckets (int): Tamaño del vocabulario para el embedding 
                (debe coincidir con el `hash_bucket_size` del dataset).
            emb_dim (int, opcional): Dimensión del vector denso para el ID.
            hidden_dims (list[int], opcional): Lista con la cantidad de neuronas 
                por capa oculta.
            dropout (float, opcional): Probabilidad de desactivación de neuronas 
                para regularización.
        """
        super().__init__()

        # Representación densa para customer_id
        self.embedding = nn.Embedding(num_hash_buckets, emb_dim)

        layers = []
        input_dim = num_numeric_features + emb_dim
        prev_dim = input_dim

        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.ReLU(),
                nn.BatchNorm1d(h),
                nn.Dropout(dropout),
            ])
            prev_dim = h

        layers.append(nn.Linear(prev_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_num: torch.Tensor, x_id: torch.Tensor) -> torch.Tensor:
        """
        Realiza el pase hacia adelante (forward pass) del modelo.

        El flujo de datos es:
        1. $x_{id} \rightarrow \text{Embedding} \rightarrow \text{emb}$
        2. $\text{input} = [x_{num} ; \text{emb}]$ (concatenación)
        3. $\text{input} \rightarrow \text{MLP} \rightarrow \text{output}$

        Args:
            x_num (torch.Tensor): Tensores numéricos de forma $(N, \text{num\_numeric\_features})$.
            x_id (torch.Tensor): Tensores de índices (hashes) de forma $(N,)$.

        Returns:
            torch.Tensor: Predicción escalar para cada muestra, de forma $(N,)$.
        """
        emb = self.embedding(x_id)
        x = torch.cat([x_num, emb], dim=1)
        return self.mlp(x).squeeze(1)

In [ ]:
@dataclass
class TrainArtifacts:
    model: nn.Module
    scaler_mean: np.ndarray
    scaler_std: np.ndarray

# 5. Training & evaluation distribuido optimizado

Esta seccion usa `run_with_torch_distributor` + `DistributedDataParallel` de PyTorch.

Cambios pensados para bajar el tiempo por epoca:

* El driver resuelve primero los prefijos S3 a archivos parquet reales (`TRAIN_FILES`, `VALID_FILES`).
* Cada rank DDP recibe un subconjunto distinto de archivos: `files[rank::world_size]`.
* Si hay workers de DataLoader, cada worker recibe otra particion local: `files_for_rank[worker_id::num_workers]`.
* El `IterableDataset` produce batches de tensores directamente. Ya no entrega una fila por iteracion.
* El bucket MD5 de `customer_id` se precalcula con Spark y se lee como `customer_id_bucket`.
* Las metricas se agregan con `all_reduce` para reportar un unico resultado global.




In [ ]:
def train_one_epoch_distributed(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> dict[str, float]:
    model.train()

    total_loss = 0.0
    total_examples = 0
    total_correct = 0

    # DDP.join permite que los ranks terminen en momentos distintos
    # cuando el particionado por archivos no produce exactamente el mismo
    # numero de batches en todos los procesos. Sin esto, un rank que termina
    # antes puede dejar colgados a los demas en el all-reduce de gradientes.
    join_context = model.join() if isinstance(model, DDP) else nullcontext()

    with join_context:
        for batch in loader:
            x_num = batch["x_num"].to(device, non_blocking=True)
            x_id = batch["x_id"].to(device, non_blocking=True)
            y = batch["y"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x_num, x_id)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                preds = (torch.sigmoid(logits) >= 0.5).float()
                correct = (preds == y).sum().item()

            bs = y.size(0)
            total_loss += loss.item() * bs
            total_examples += bs
            total_correct += correct

    total_loss = reduce_sum(total_loss, device)
    total_examples = reduce_sum(total_examples, device)
    total_correct = reduce_sum(total_correct, device)

    return {
        "loss": total_loss / max(total_examples, 1),
        "accuracy": total_correct / max(total_examples, 1),
    }


@torch.no_grad()
def evaluate_model_distributed(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    threshold: float = 0.5,
) -> dict[str, float]:
    model.eval()

    tp = tn = fp = fn = 0

    for batch in loader:
        x_num = batch["x_num"].to(device, non_blocking=True)
        x_id = batch["x_id"].to(device, non_blocking=True)
        y = batch["y"].to(device, non_blocking=True)

        logits = model(x_num, x_id)
        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).long()
        y_int = y.long()

        tp += int(((y_int == 1) & (preds == 1)).sum().item())
        tn += int(((y_int == 0) & (preds == 0)).sum().item())
        fp += int(((y_int == 0) & (preds == 1)).sum().item())
        fn += int(((y_int == 1) & (preds == 0)).sum().item())

    tp = int(reduce_sum(tp, device))
    tn = int(reduce_sum(tn, device))
    fp = int(reduce_sum(fp, device))
    fn = int(reduce_sum(fn, device))

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "precision": float(precision),
        "recall": float(recall),
        "accuracy": float(accuracy),
        "f1_score": float(f1),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


In [ ]:
def train_lifetime_distributed(
    train_files: list[str],
    valid_files: list[str],
    scaler_mean: np.ndarray,
    scaler_std: np.ndarray,
    pos_weight_value: float,
    hidden_dims: list[int],
    dropout: float,
    learning_rate: float,
    weight_decay: float,
    epochs: int,
    seed: int,
    artifact_path: str,
) -> dict:
    configure_worker_runtime()
    print("RANK:", os.environ.get("RANK"), flush=True)
    print("LOCAL_RANK:", os.environ.get("LOCAL_RANK"), flush=True)
    print("WORLD_SIZE:", os.environ.get("WORLD_SIZE"), flush=True)
    print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"), flush=True)

    rank, local_rank, world_size = setup_distributed(backend="gloo")
    print(f"DDP setup: rank={rank}, local_rank={local_rank}, world_size={world_size}", flush=True)
    print(f"SSL_CERT_FILE={os.environ.get('SSL_CERT_FILE')}", flush=True)

    try:
        set_seed(seed + rank)
        device = torch.device("cpu")

        if len(train_files) < world_size:
            raise ValueError(
                f"Hay {len(train_files)} archivos train para world_size={world_size}. "
                "DDP necesita suficientes archivos para repartir trabajo. "
                "Reduce NUM_PROCESSES o usa mas particiones/archivos."
            )

        train_loader = make_train_loader(
            train_files=train_files,
            rank=rank,
            world_size=world_size,
        )

        base_model = TabularNN(
            num_numeric_features=len(NUMERIC_COLS),
            num_hash_buckets=HASH_BUCKET_SIZE,
            emb_dim=EMBEDDING_DIM,
            hidden_dims=hidden_dims,
            dropout=dropout,
        ).to(device)

        # CPU DDP: no usar device_ids ni output_device.
        model = DDP(base_model) if world_size > 1 else base_model

        pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )

        history = []
        for epoch in range(1, epochs + 1):
            metrics = train_one_epoch_distributed(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                criterion=criterion,
                device=device,
            )
            if rank == 0:
                row = {"epoch": epoch, **metrics}
                history.append(row)
                print(
                    f"Epoch {epoch:02d} | "
                    f"train_loss={metrics['loss']:.6f} | "
                    f"train_acc={metrics['accuracy']:.6f}",
                    flush=True,
                )

        valid_loader = make_valid_loader(
            valid_files=valid_files,
            rank=rank,
            world_size=world_size,
        )

        valid_metrics = evaluate_model_distributed(
            model=model,
            loader=valid_loader,
            device=device,
            threshold=0.5,
        )

        if rank == 0:
            torch.save(
                {
                    "model_state_dict": base_model.state_dict(),
                    "numeric_cols": NUMERIC_COLS,
                    "target_col": TARGET_COL,
                    "id_col": ID_COL,
                    "id_bucket_col": ID_BUCKET_COL,
                    "hash_bucket_size": HASH_BUCKET_SIZE,
                    "embedding_dim": EMBEDDING_DIM,
                    "hidden_dims": hidden_dims,
                    "dropout": dropout,
                    "scaler_mean": scaler_mean,
                    "scaler_std": scaler_std,
                    "pos_weight_value": pos_weight_value,
                    "history": history,
                    "valid_metrics": valid_metrics,
                },
                artifact_path,
            )

            return {
                "status": "ok",
                "history": history,
                "valid_metrics": valid_metrics,
                "artifact_path": artifact_path,
            }

        return {"status": "worker_done"}

    finally:
        cleanup_distributed()


## 5.1 Precomputo streaming en el driver

Estas operaciones se hacen antes de lanzar `TorchDistributor`, para no repetir el calculo completo de medias, desviaciones y balance de clases en cada proceso distribuido.

In [ ]:
set_seed(SEED)

if RUN_SPARK_PREPARE:
    print("Preparando datasets con Spark...")
    _, _, scaler_mean, scaler_std = prepare_lifetime_datasets_with_spark()
else:
    print("RUN_SPARK_PREPARE=False. Se asume que los Parquet preparados ya existen y YA estan z-scoreados.")
    print("Calculando estadisticas raw de train solo para guardarlas en el artefacto de inferencia...")
    scaler_mean, scaler_std = compute_streaming_stats(
        paths=PATHS,
        numeric_cols=NUMERIC_COLS,
        batch_size_rows=BATCH_SIZE_ROWS,
        fill_null_value=FILL_NULL_VALUE,
        region=AWS_REGION,
    )

print("Resolviendo archivos parquet preparados de train...")
TRAIN_FILES = expand_s3_paths([PREPARED_TRAIN_PATH], region=AWS_REGION)
print(f"Archivos train preparados encontrados: {len(TRAIN_FILES)}")

print("Resolviendo archivos parquet preparados de validacion...")
VALID_FILES = expand_s3_paths([PREPARED_VALID_PATH], region=AWS_REGION)
print(f"Archivos validacion preparados encontrados: {len(VALID_FILES)}")

if len(TRAIN_FILES) < NUM_PROCESSES:
    print(
        f"ADVERTENCIA: solo hay {len(TRAIN_FILES)} archivos train preparados para {NUM_PROCESSES} procesos. "
        "Aumenta SPARK_REPARTITIONS_TRAIN o reduce NUM_PROCESSES."
    )

print("Calculando class imbalance SOLO con train preparado...")
n_neg, n_pos = compute_class_balance(
    paths=TRAIN_FILES,
    target_col=TARGET_COL,
    batch_size_rows=BATCH_SIZE_ROWS,
    region=AWS_REGION,
)

print(f"Negativos: {n_neg}")
print(f"Positivos: {n_pos}")

if n_pos == 0:
    raise ValueError("No hay ejemplos positivos en train.")

pos_weight_value = n_neg / max(n_pos, 1)
print(f"pos_weight: {pos_weight_value:.6f}")

## 5.2 Entrenamiento distribuido con `run_with_torch_distributor`

In [ ]:

import traceback
import sys


def train_lifetime_distributed_debug(*args):
    """Wrapper de depuracion para TorchDistributor.

    TorchDistributor dentro de Spark Barrier suele esconder el error real detras de:
    RuntimeError: TorchDistributor failed during training.

    Este wrapper imprime diagnostico y, si falla cualquier rank, escribe el traceback completo
    en DEBUG_LOG_PATH para poder leerlo desde el driver despues del fallo.
    """
    import os
    import sys
    import traceback

    configure_worker_runtime()
    rank = os.environ.get("RANK", os.environ.get("OMPI_COMM_WORLD_RANK", "unknown"))

    diagnostics = format_worker_diagnostics(
        prefix="ENTRANDO A train_lifetime_distributed_debug"
    )
    print("=" * 100, flush=True)
    print(diagnostics, flush=True)
    print("=" * 100, flush=True)

    try:
        return train_lifetime_distributed(*args)
    except Exception as e:
        tb = traceback.format_exc()
        log_text = f"""
ERROR REAL DENTRO DE train_lifetime_distributed_debug

{format_worker_diagnostics(prefix='DIAGNOSTICO DEL WORKER')}

Exception:
{type(e).__name__}: {str(e)}

Traceback:
{tb}
"""
        print("=" * 100, flush=True)
        print(log_text, flush=True)
        print("=" * 100, flush=True)
        sys.stdout.flush()
        sys.stderr.flush()
        write_rank_debug_log(log_text, rank=rank, region=AWS_REGION)
        raise


In [ ]:

print(f"DEBUG_LOG_PATH: {DEBUG_LOG_PATH}")

train_args = (
    TRAIN_FILES,
    VALID_FILES,
    scaler_mean,
    scaler_std,
    pos_weight_value,
    [128, 64],
    0.3,
    LEARNING_RATE,
    WEIGHT_DECAY,
    EPOCHS,
    SEED,
    MODEL_ARTIFACT_PATH,
)

distributed_result = run_with_torch_distributor(
    train_fn=train_lifetime_distributed_debug,
    num_processes=NUM_PROCESSES,
    local_mode=False,
    use_gpu=False,
    train_args=train_args,
)

distributed_result


## 5.3 Lectura manual de logs por rank si TorchDistributor falla

Si la celda de entrenamiento falla con el error generico de Spark Barrier, ejecuta esta celda para listar y leer los logs escritos por cada rank en S3.


In [ ]:

def list_debug_logs(debug_log_path: str = DEBUG_LOG_PATH, region: str = AWS_REGION) -> list[str]:
    fs = get_s3_filesystem(region=region)
    bucket, key = parse_s3_uri(debug_log_path)
    selector = pafs.FileSelector(f"{bucket}/{key.rstrip('/')}/", recursive=True)
    infos = fs.get_file_info(selector)
    return sorted(
        f"s3://{info.path}"
        for info in infos
        if info.type == pafs.FileType.File and info.path.endswith(".log")
    )


def read_s3_text(s3_uri: str, region: str = AWS_REGION) -> str:
    fs = get_s3_filesystem(region=region)
    bucket, key = parse_s3_uri(s3_uri)
    with fs.open_input_file(f"{bucket}/{key}") as f:
        return f.read().decode("utf-8")


logs = list_debug_logs()
print(f"Logs encontrados: {len(logs)}")
for log in logs[-10:]:
    print(log)

if logs:
    print("\n===== ULTIMO LOG =====\n")
    print(read_s3_text(logs[-1]))


## 5.3 Metricas finales

In [ ]:
print("Valid metrics:")
for k, v in distributed_result["valid_metrics"].items():
    print(f"{k}: {v}")

print("Artifact path:", distributed_result["artifact_path"])

# 6. Notas de implementacion

* Spark se usa antes del entrenamiento para preparar `train_prepared/` y `valid_prepared/`, incluyendo casteos, `fillna`, reparticionado y `customer_id_bucket` con MD5.
* `ArrowParquetBatchDataset` es lazy: lee Parquet preparado con PyArrow en streaming y produce batches de tensores.
* No se usa `DistributedSampler` porque `IterableDataset` reparte los archivos internamente por `rank`, `world_size`, `worker_id` y `num_workers`.
* No hagas sharding previo tipo `train_files[rank::world_size]`; el dataset ya hace ese reparto. Si se hiciera antes, partiríamos la data dos veces.
* Para EMR CPU se usa `gloo`, `use_gpu=False` y `local_mode=False`.
* La evaluacion tambien se reparte por rank y agrega la matriz de confusion con `all_reduce`.
